In [14]:
import numpy as np
import re
import os

In [15]:
input_folder = os.path.join("Input",'orca_movie')
out_folder = os.path.join("Output",'orca_movie')

In [16]:
def align_matrix(coord,element):
    atomic_symbols = {
    1: "H", 2: "He",
    3: "Li", 4: "Be", 5: "B", 6: "C", 7: "N", 8: "O", 9: "F", 10: "Ne",
    11: "Na", 12: "Mg", 13: "Al", 14: "Si", 15: "P", 16: "S", 17: "Cl", 18: "Ar",
    19: "K", 20: "Ca", 21: "Sc", 22: "Ti", 23: "V", 24: "Cr", 25: "Mn", 26: "Fe",
    27: "Co", 28: "Ni", 29: "Cu", 30: "Zn", 31: "Ga", 32: "Ge", 33: "As", 34: "Se",
    35: "Br", 36: "Kr", 37: "Rb", 38: "Sr", 39: "Y", 40: "Zr", 41: "Nb", 42: "Mo",
    43: "Tc", 44: "Ru", 45: "Rh", 46: "Pd", 47: "Ag", 48: "Cd", 49: "In", 50: "Sn",
    51: "Sb", 52: "Te", 53: "I", 54: "Xe", 55: "Cs", 56: "Ba", 57: "La", 58: "Ce",
    59: "Pr", 60: "Nd", 61: "Pm", 62: "Sm", 63: "Eu", 64: "Gd", 65: "Tb", 66: "Dy",
    67: "Ho", 68: "Er", 69: "Tm", 70: "Yb", 71: "Lu", 72: "Hf", 73: "Ta", 74: "W",
    75: "Re", 76: "Os", 77: "Ir", 78: "Pt", 79: "Au", 80: "Hg", 81: "Tl", 82: "Pb",
    83: "Bi", 84: "Po", 85: "At", 86: "Rn", 87: "Fr", 88: "Ra", 89: "Ac", 90: "Th",
    91: "Pa", 92: "U", 93: "Np", 94: "Pu", 95: "Am", 96: "Cm", 97: "Bk", 98: "Cf",
    99: "Es", 100: "Fm", 101: "Md", 102: "No", 103: "Lr", 104: "Rf", 105: "Db", 106: "Sg",
    107: "Bh", 108: "Hs", 109: "Mt", 110: "Ds", 111: "Rg", 112: "Cn", 113: "Nh", 114: "Fl",
    115: "Mc", 116: "Lv", 117: "Ts", 118: "Og"
    }
    symbol_to_Z = {v: k for k, v in atomic_symbols.items()}
    z_element = np.zeros(len(element))
    for i,atom in enumerate(element):
        z_element[i] = symbol_to_Z[atom]
    if element[1] == "N":
        byp = coord[0:21]
    elif element[1] == "C":
        byp = coord[[0] + list(range(8, 26)), :]
    c = byp.mean(axis=0)
    q = byp - c

    C = q.T @ q
    vals, vecs = np.linalg.eigh(C)
    n = vecs[:, np.argmin(vals)]
    n = n / np.linalg.norm(n)

    for i,atom in enumerate(coord):
        I_n_minus = sum(z_element[i]*(atom-n)**2)
        I_n_plus = sum(z_element[i]*(atom+n)**2)
    if I_n_plus > I_n_minus:
        n = n
    r_ref = coord[0]
    x0 = r_ref - c
    x = x0 - np.dot(x0, n) * n
    x_hat = x / np.linalg.norm(x)

    y_hat = np.cross(n, x_hat)
    y_hat = y_hat / np.linalg.norm(y_hat)

    R = np.vstack([x_hat, y_hat, n])
    return R, c
def align(coord,R,c):
    r_shift = coord - c
    coords_new = (R @ r_shift.T).T
    coords_new = coords_new - coords_new[0]
    return coords_new

In [17]:
for filename in os.listdir(input_folder):
    with open(os.path.join(input_folder,filename),'r') as f:
        lines = f.readlines()
        j = 2
        n_lines = sum(1 for _ in lines)
        for i,line in enumerate(lines):
            if i == 0:
                atoms = int(line)
                n_frames = n_lines//(atoms+2)
                imgs = np.zeros((n_frames, atoms, 3))
                elements = []
                continue
            if i % (atoms+2) == j:
                parts = line.split()
                if len(parts) == 4:
                    x, y, z = map(float, parts[1:4])
                    imgs[i // (atoms + 2), j - 2, :] = [x, y, z]
                    j += 1
                if len(elements) < atoms:
                    elements.append(parts[0])
            if j == atoms+2:
                j = 2

In [18]:
quality = 60
n_frames = n_frames-1
smooth_matrix = np.zeros((n_frames*quality+1,atoms,3))
R, c = align_matrix(imgs[0,:,:],elements)

In [19]:
for i in range(n_frames+1):
    if i == n_frames:
        smooth_matrix[-1,:,:] = imgs[-1,:,:]
        break
    delta = (imgs[i+1,:] - imgs[i,:])/quality
    for z in range(quality):
        smooth_matrix[i*quality+z,:,:] = imgs[i,:] + delta*z
        smooth_matrix[i*quality+z,:,:] = align(smooth_matrix[i*quality+z,:,:],R,c)

In [20]:
print(hola)

NameError: name 'hola' is not defined

In [ ]:
delta = np.zeros((3))
for i in range(n_frames+1):
    if i == n_frames:
        smooth_matrix[-1,:,:] = imgs[-1,:,:]
        break
    for z in range(quality):
        for j in range(atoms):
            for k in range(3):
                delta[k] = (imgs[i,j,k] - imgs[i + 1,j,k])/quality
                smooth_matrix[i*quality+z,j,k] = imgs[i,j,k] - delta[k]*z

In [21]:
with open(os.path.join(out_folder, "test.xyz"), "w") as f:
    n_frames_smooth = n_frames * quality +1

    for frame in range(n_frames_smooth):
        f.write(f"{atoms}\n")
        f.write(f"Interpolated frame {frame}\n")

        for j in range(atoms):
            x, y, z = smooth_matrix[frame, j]   # <-- correct indexing
            f.write(f"{elements[j]:<3}  {x:15.6f}  {y:15.6f}  {z:15.6f}\n")

In [ ]:
i=60*28
frame_tot = 10
frame_int = (i//atoms)%quality
n_frame_tot = i//(quality*atoms)
print(len(imgs)//atoms)
print(n_frame_tot)
print((i//atoms)%quality)